In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import datetime as dt
from datetime import timedelta

🛢️ Energia
Commodity	Ticker (Yahoo Finance)	Descrizione
Petrolio WTI	CL=F	West Texas Intermediate Crude Oil
Petrolio Brent	BZ=F	Brent Crude Oil
Gas Naturale	NG=F	Natural Gas
Benzina RBOB	RB=F	Gasoline

🥇 Metalli Preziosi
Commodity	Ticker (Yahoo Finance)	Descrizione
Oro	GC=F	Gold Futures
Argento	SI=F	Silver Futures
Platino	PL=F	Platinum
Palladio	PA=F	Palladium

🏗️ Metalli Industriali
Commodity	Ticker (Yahoo Finance)	Descrizione
Rame	HG=F	Copper
Alluminio	(non disponibile diretto su Yahoo)	Spesso via LME

🌾 Agricoli / Soft Commodities
Commodity	Ticker (Yahoo Finance)	Descrizione
Grano	ZW=F	Wheat
Mais	ZC=F	Corn
Soia	ZS=F	Soybeans
Zucchero	SB=F	Sugar
Caffè	KC=F	Coffee
Cotone	CT=F	Cotton
Cacao	CC=F	Cocoa
Suini magri	HE=F	Lean Hogs
Bovine da carne	LE=F	Live Cattle

In [ ]:
commodity = ["CL=F", "BZ=F", "NG=F", "RB=F", "GC=F", "SI=F", "PL=F", "PA=F", "HG=F", "ZW=F" , "ZC=F", "ZS=F"]


"HG=F", "ZW=F" , "GC=F", 

end = dt.datetime.now()
start = end - dt.timedelta(days = 300)

return_commodity = yf.download(commodity, start, end, interval = "60m")["Close"].pct_change().dropna()

C:\Users\miche\AppData\Local\Temp\ipykernel_40252\2686411727.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return_commodity = yf.download(commodity, start, end, interval = "60m")["Close"].pct_change().dropna()
[*********************100%***********************]  12 of 12 completed
C:\Users\miche\AppData\Local\Temp\ipykernel_40252\2686411727.py:6: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_commodity = yf.download(commodity, start, end, interval = "60m")["Close"].pct_change().dropna()


FREQUENCY: DAILY VOL

In [7]:
return_commodity

Ticker,BZ=F,CL=F,GC=F,HG=F,NG=F,PA=F,PL=F,RB=F,SI=F,ZC=F,ZS=F,ZW=F
Datetime,,,,,,,,,,,,
2024-06-10 01:00:00+00:00,0.001381,0.001191,0.000389,-0.001564,-0.000671,0.001616,0.002563,0.000752,0.005609,0.044975,-0.015705,0.000799
2024-06-10 02:00:00+00:00,0.000627,0.000661,-0.000648,0.000224,0.000336,-0.001613,-0.002250,-0.000042,0.000676,-0.000531,0.016602,-0.000799
2024-06-10 03:00:00+00:00,0.000626,0.000396,-0.000562,-0.002684,0.001678,0.000539,-0.001230,0.000626,-0.002872,-0.040936,-0.000424,-0.001199
2024-06-10 04:00:00+00:00,-0.000250,-0.000396,-0.000173,0.002692,0.000000,0.003767,0.002052,-0.000209,0.004574,-0.000554,0.000000,0.001200
2024-06-10 05:00:00+00:00,-0.000376,-0.000528,0.000433,0.001119,-0.001340,-0.000536,0.001229,0.000584,0.001349,-0.000555,-0.000424,-0.000400
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-14 04:00:00+00:00,-0.001839,0.002630,0.002943,-0.005264,0.035003,0.006594,-0.007266,-0.000684,0.010365,-0.001819,0.001241,0.000000
2025-07-14 05:00:00+00:00,-0.000992,-0.001166,-0.000978,-0.002511,0.001166,-0.009634,-0.004583,-0.000319,-0.001266,-0.001215,0.000248,0.000000
2025-07-14 06:00:00+00:00,0.000851,0.001167,0.003293,-0.000360,0.007863,-0.004280,-0.007422,-0.000228,-0.001648,0.001825,-0.000743,0.001376


In [6]:
def get_day(dataset, i):
    dataset = dataset.reset_index()
    date = dataset.iloc[i,0]
    if isinstance(date, dt.datetime):
        date = str(date)
    date = date[0:10]

    return date

In [7]:
volatility = pd.DataFrame([])
Date = pd.DataFrame([])


for tick in commodity:
    vol_day = []
    days = []
    std = []
    ret_day = []
    condition = False

    for i in range(len(return_commodity[tick])):
        date = get_day(return_commodity, i)
        
        if i == 0:
            ret = return_commodity[tick].iloc[i]
            ret_day = np.append(ret_day, ret)
            days = np.append(days, date)

            continue
        
        prev_date = get_day(return_commodity, i-1)

        if date == prev_date:
            ret = return_commodity[tick].iloc[i]
            #print(ret)
            ret_day = np.append(ret_day, ret)

             
            if condition == True:
                ret = return_commodity[tick].iloc[i-1]
                ret_day = np.append(ret_day,ret)
                condition == False
        else:
            std = ret_day.std()
            vol_day = np.append(vol_day, std)
            days = np.append(days, date)
            #print(ret_day)
            condition == True

        if i == len(return_commodity[tick]) - 1:
            ret = return_commodity[tick].iloc[i]
            ret_day = np.append(ret_day, ret)

            std = ret_day.std()
            vol_day = np.append(vol_day, std)
            #days = np.append(days, date)

    vol_day = pd.DataFrame(vol_day)
    volatility = pd.concat([volatility, vol_day], axis = 1, ignore_index= True)
    #Date = np.append(Date, days)



In [8]:
days = pd.DataFrame(days)
col = commodity + ["Date"]
vol_commodity_daily = pd.concat([volatility,days], axis = 1, ignore_index=True)
vol_commodity_daily.columns = col
vol_commodity_daily = vol_commodity_daily.set_index("Date")
vol_commodity_daily

,CL=F,BZ=F,NG=F,RB=F,GC=F,SI=F,PL=F,PA=F,HG=F,ZW=F,ZC=F,ZS=F
Date,,,,,,,,,,,,
2024-12-26,0.001614,0.003598,0.015804,0.002282,0.001347,0.001402,0.001735,0.004460,0.001557,0.001816,0.001694,0.002375
2024-12-27,0.001925,0.002834,0.016840,0.002298,0.001380,0.001698,0.003270,0.003739,0.001528,0.002145,0.001418,0.001891
2024-12-29,0.001925,0.002834,0.016840,0.002298,0.001380,0.001698,0.003270,0.003739,0.001528,0.002145,0.001418,0.001891
2024-12-30,0.001999,0.002552,0.018021,0.002517,0.001648,0.002709,0.003522,0.004138,0.001533,0.002606,0.001960,0.002273
2024-12-31,0.002226,0.002592,0.016973,0.002497,0.001594,0.002432,0.003311,0.004099,0.001603,0.002448,0.001780,0.002298
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-15,0.004342,0.005660,0.010207,0.005002,0.002288,0.003776,0.004290,0.005120,0.005065,0.007431,0.005662,0.004632
2025-10-16,0.004336,0.005650,0.010189,0.005085,0.002293,0.003792,0.004306,0.005137,0.005059,0.007414,0.005649,0.004706
2025-10-17,0.004333,0.005641,0.010170,0.005326,0.002318,0.003819,0.004328,0.005160,0.005054,0.007398,0.005638,0.004775


In [9]:
daily_commodity_vol = vol_commodity_daily.to_csv("daily_commodity_vol.csv")

WEEKLY VOL

In [54]:
commodity = ["CL=F", "BZ=F", "NG=F", "RB=F", "GC=F", "SI=F", "PL=F", "PA=F", "HG=F", "ZW=F" , "ZC=F", "ZS=F"]

end = dt.datetime.now()
start = end - dt.timedelta(days = 3000)

return_commodity_daily = yf.download(commodity, start, end)["Close"].pct_change().dropna()

C:\Users\miche\AppData\Local\Temp\ipykernel_40252\572408041.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return_commodity_daily = yf.download(commodity, start, end)["Close"].pct_change().dropna()
[*********************100%***********************]  12 of 12 completed
C:\Users\miche\AppData\Local\Temp\ipykernel_40252\572408041.py:6: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_commodity_daily = yf.download(commodity, start, end)["Close"].pct_change().dropna()


In [ ]:
return_commodity_daily

Ticker,BZ=F,CL=F,GC=F,HG=F,NG=F,PA=F,PL=F,RB=F,SI=F,ZC=F,ZS=F,ZW=F
Date,,,,,,,,,,,,
2024-02-29,-0.000717,-0.003565,0.006247,0.001829,-0.013263,0.016885,-0.000908,0.014663,0.011423,0.006050,-0.005071,0.011384
2024-03-01,-0.000837,0.021850,0.020140,0.005346,-0.013441,0.020675,0.004887,0.134574,0.021354,-0.008419,0.013073,-0.030303
2024-03-04,-0.008977,-0.015381,0.014759,-0.000778,0.044142,0.012699,0.018435,-0.010978,0.027171,0.012735,0.003500,0.006250
2024-03-05,-0.009179,-0.007493,0.007461,-0.002077,0.021399,-0.027257,-0.019101,-0.020459,-0.000673,-0.009581,-0.005449,-0.031500
2024-03-06,0.011214,0.012540,0.007874,0.006894,-0.014308,0.105796,0.031020,0.008331,0.021420,0.010883,-0.000438,-0.044892
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-14,-0.014687,-0.013280,0.007326,-0.023527,-0.028865,0.039751,-0.008685,-0.008244,0.003670,0.005478,-0.001240,0.007046
2025-10-15,-0.007694,-0.007325,0.009230,-0.001707,-0.003963,-0.000191,0.008217,0.003172,0.015085,0.009080,0.000000,-0.002999
2025-10-16,-0.013730,-0.013901,0.024731,-0.002816,-0.025862,0.054745,0.039672,-0.012375,0.038181,0.011998,0.004223,0.007519


In [55]:
weekly_volatility_commodity = pd.DataFrame([])

for tick in commodity:

    ret = return_commodity_daily[[tick]]
    ret.index = pd.to_datetime(return_commodity_daily.index)

    vol = ret.resample("W").std()

    weekly_volatility_commodity = pd.concat([weekly_volatility_commodity, vol], axis = 1, ignore_index = True)
    #volatility_commodity.columns = commodity

weekly_volatility_commodity.dropna(inplace = True)
#weekly_volatility_commodity.drop(weekly_volatility_commodity.columns[-1], axis = 1, inplace = True)
weekly_volatility_commodity.columns = commodity
weekly_volatility_commodity



,CL=F,BZ=F,NG=F,RB=F,GC=F,SI=F,PL=F,PA=F,HG=F,ZW=F,ZC=F,ZS=F
Date,,,,,,,,,,,,
2017-08-13,0.010686,0.009667,0.014038,0.007098,0.006106,0.011848,0.003462,0.009361,0.008313,0.022859,0.022434,0.017625
2017-08-20,0.021477,0.022882,0.011350,0.018453,0.005826,0.014154,0.009720,0.010614,0.012914,0.014747,0.010617,0.008298
2017-08-27,0.018098,0.013684,0.016342,0.019826,0.003986,0.004093,0.003240,0.007927,0.008921,0.013429,0.004441,0.004525
2017-09-03,0.019946,0.019221,0.014855,0.118723,0.006234,0.010325,0.008539,0.025949,0.005849,0.018334,0.021222,0.009354
2017-09-10,0.025924,0.014716,0.019990,0.009539,0.006931,0.005904,0.006047,0.019872,0.018107,0.020829,0.012414,0.011457
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-21,0.013887,0.011353,0.033692,0.015062,0.007834,0.013359,0.015184,0.014124,0.011108,0.022825,0.028357,0.011424
2025-09-28,0.010548,0.010744,0.021516,0.009702,0.012055,0.017538,0.026841,0.019077,0.017849,0.017799,0.007904,0.007004
2025-10-05,0.015168,0.014311,0.073244,0.017831,0.007641,0.024472,0.021010,0.017732,0.016143,0.012246,0.010020,0.009377


In [65]:
weekly_vol_commodity = weekly_volatility_commodity.to_csv("weekly_vol_commodity.csv")

FREQUENCY: MONTHLY VOL

In [62]:
commodity = ["CL=F", "BZ=F", "NG=F", "RB=F", "GC=F", "SI=F", "PL=F", "PA=F", "HG=F", "ZW=F" , "ZC=F", "ZS=F"]

end = dt.datetime.now()
start = end - dt.timedelta(days = 4000)


return_commodity_d = yf.download(commodity, start, end)["Close"].pct_change().dropna()

C:\Users\miche\AppData\Local\Temp\ipykernel_40252\3618306955.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  return_commodity_d = yf.download(commodity, start, end)["Close"].pct_change().dropna()
[*********************100%***********************]  12 of 12 completed
C:\Users\miche\AppData\Local\Temp\ipykernel_40252\3618306955.py:7: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_commodity_d = yf.download(commodity, start, end)["Close"].pct_change().dropna()


In [ ]:
monthly_volatility_commodity = pd.DataFrame([])

for tick in commodity:
    ret = return_commodity_d[[tick]]
    ret.index = pd.to_datetime(return_commodity_d.index)

    vol = ret.resample("ME").std()

    monthly_volatility_commodity = pd.concat([monthly_volatility_commodity, vol], axis = 1, ignore_index = True)

monthly_volatility_commodity.columns = commodity

monthly_volatility_commodity

,CL=F,BZ=F,NG=F,RB=F,GC=F,SI=F,PL=F,PA=F,HG=F,ZW=F,ZC=F,ZS=F
Date,,,,,,,,,,,,
2014-11-30,0.030377,0.028238,0.039710,0.023188,0.008875,0.023032,0.009977,0.011801,0.012780,0.016241,0.015763,0.019850
2014-12-31,0.026617,0.020832,0.037785,0.019593,0.014119,0.028540,0.013304,0.011249,0.010451,0.022871,0.014279,0.011809
2015-01-31,0.038476,0.031964,0.047769,0.028893,0.011452,0.023911,0.012072,0.017351,0.018143,0.010805,0.014858,0.013956
2015-02-28,0.040715,0.039450,0.029477,0.035359,0.008026,0.018011,0.009839,0.012985,0.013180,0.018936,0.014022,0.011827
2015-03-31,0.027877,0.027672,0.026395,0.025713,0.008544,0.016205,0.012537,0.013954,0.016315,0.022189,0.015832,0.009826
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-30,0.034241,0.031310,0.050499,0.024009,0.010843,0.017228,0.028382,0.027088,0.013917,0.020145,0.011961,0.009504
2025-07-31,0.017399,0.015310,0.028342,0.014926,0.007783,0.014959,0.025642,0.028958,0.056785,0.014127,0.014215,0.010473
2025-08-31,0.014232,0.014843,0.028717,0.014806,0.008809,0.012319,0.011815,0.019319,0.009426,0.010985,0.014477,0.010500


In [66]:
monthly_vol_commodity = monthly_volatility_commodity.to_csv("monthly_vol_commodity.csv")